# Daily limit-distance experiments

Compare fixed percentages below the preceding close using both one-session outcomes and recurring monthly cash lots.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px

from retail_sp500.daily_data import daily_data_summary, load_or_fetch_twelve_data_daily

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.6f}")

from retail_sp500.limit_orders import evaluate_limit_discount_grid
from retail_sp500.limit_portfolio import evaluate_recurring_limit_grid

In [ ]:
SYMBOL = "SPY"
START_DATE = "2007-06-01"
CACHE_PATH = Path("data/processed/spy_daily_1day.csv")
REFRESH = False

daily = load_or_fetch_twelve_data_daily(
    os.getenv("TWELVE_DATA_API_KEY"),
    cache_path=CACHE_PATH,
    refresh=REFRESH,
    symbol=SYMBOL,
    start_date=START_DATE,
)

source = daily_data_summary(daily, symbol=SYMBOL)
print(source)
assert source["interval"] == "1day"
assert daily.index.max() <= pd.Timestamp.today().normalize()
daily.tail()

## Parameter grid

In [ ]:
DISCOUNTS = np.arange(0.0, 0.0501, 0.001)
WAIT_HORIZONS = [1, 3, 5, 10, 20]

one_session = evaluate_limit_discount_grid(daily, DISCOUNTS)
recurring = pd.concat(
    [
        evaluate_recurring_limit_grid(
            daily,
            DISCOUNTS,
            max_wait_sessions=wait,
            initial_cash=100_000,
            monthly_contribution=1_000,
        ).assign(wait_horizon=wait)
        for wait in WAIT_HORIZONS
    ],
    ignore_index=True,
)

one_session.sort_values("mean_one_session_excess_vs_open", ascending=False).head(10)

## Practical recurring-purchase candidates

Unfilled lots are repriced each session and forced at that session close when their wait horizon expires.

In [ ]:
best_by_wait = recurring.loc[
    recurring.groupby("wait_horizon")["ending_excess_value"].idxmax()
].sort_values("wait_horizon")

best_by_wait[[
    "wait_horizon",
    "discount",
    "ending_excess_value",
    "ending_excess_pct_of_contributions",
    "limit_fill_rate",
    "forced_fill_rate",
    "average_wait_sessions",
    "weighted_execution_savings_vs_immediate_open",
]]

In [ ]:
px.line(
    recurring,
    x="discount",
    y="ending_excess_pct_of_contributions",
    color="wait_horizon",
    title="Recurring-purchase excess value by limit distance and expiry",
    labels={"discount": "Limit below preceding close"},
).show()

In [ ]:
px.line(
    recurring,
    x="discount",
    y="forced_fill_rate",
    color="wait_horizon",
    title="How often the limit misses and requires expiry execution",
).show()

## Interpretation boundary

This is still an execution-only comparison. Dividends, cash yield, spread, fees, taxes, and SGD/USD effects are not included.